In [ ]:
import os
import sys
from IPython.display import display, Markdown


# Add the current directory to sys.path
print('Current path:')
print(os.getcwd())

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'data')))

try:
    from multiconductor import MulticonductorTransmissionLine as Multiconductor
    print("Imports successfully done!")
except ImportError as e:
    print(f"Imports error: {e}")

# Create a new list without duplicate entries using a set and update sys.path
sys.path = list(dict.fromkeys(sys.path))

In [ ]:
# Simple test to see if the Markdown function works
test_markdown = Markdown(r"Markdown format test: $\varepsilon_0=8.854 \times 10^{-12} F/m$ and $\mu_0=4\pi \times 10^{-7} H/m$")
display(test_markdown)

# Derivation of generalized earth-return impedance and admittance formulas

The generalized earth-return impedance and admittance formulas are derived based on the Quasi-TEM solution for electromagnetic field components.  

References:  
1. XUE, Haoyan. _General Formulation and Accurate Evaluation of Earth-Return Parameters for Overhead / Underground Cables_. PhD thesis, Department of Electrical Engineering, École Polytechnique de Montréal, Université de Montréal, August 2018.  
2. MARTINEZ-VELASCO, Juan A. _Power System Transients_. Parameter Determination”, CRC Press, 2010.
3. AMETANI, Akihiro; OHNO, Teruo; NAGAOKA, Naoto. _Cable System Transients:_ Theory, Modeling and Simulation, Wiley-IEEE Press, 2015.

## Quasi-TEM solution

An overhead line with $N$ conductors is located above a homogenous earth, as illustrated in Figure 1.

<figure>
    <img src="imagens/ohtl_general_configuration.png" alt="Um exemplo de imagem" />
    <figcaption>Figure 1. Configuration of an overhead line with N conductors above a homogenous earth [1].</figcaption>
</figure>

The air is the medium in the region $z>0$, and characterized by permeability $\mu_0$ and permittivity $\varepsilon_0$. The region of $z<0$ is the lossy earth, and it is characterized by permeability $\mu_1$, permittivity $\varepsilon_1$ and conductivity $\sigma_1$.  

It is assumed that the radius of each conductor is small compared with the height and separation. Thus, a filament current approximation can be adopted where all the conductor currents are only expressed by the axial component. 

The earth-return impedance $\underline{Z}_{e(nm)} \left( d_n, h_n, d_m, h_m \right)$ and potential coefficient $\underline{P}_{e(nm)} \left( d_n, h_n, d_m, h_m \right)$ based on quasi-TEM assumption between conductors $n$ and $m$.

$$
\underline{Z}_{e(nm)} \left( d_n, h_n, d_m, h_m \right) = \frac{j \omega \mu_0}{2\pi} \left\{ \ln \left[ \frac{D_{nm} \left( d_n, h_n, d_m, h_m \right)}{d_{nm} \left( d_n, h_n, d_m, h_m \right)} \right] + 2\underline{\underline{S}}_{1} \left( d_n, h_n, d_m, h_m \right) \right\}
$$

<p style="margin-bottom: 40px;"></p>

$$
\underline{P}_{e(nm)} \left( d_n, h_n, d_m, h_m \right) = \frac{1}{2 \pi \varepsilon_0} \left\{ \ln \left[ \frac{D_{nm} \left( d_n, h_n, d_m, h_m \right)}{d_{nm} \left( d_n, h_n, d_m, h_m \right)} \right] + 2\underline{\underline{S}}_{2} \left( d_n, h_n, d_m, h_m \right) \right\}
$$

The self-elements are calculated by adopting $h_n = h_m$ and $d_n - d_m = {r_n}$. The mutual elements are calculated at the center of the two conductors.

$\underline{\underline{S}}_{1} \left( y, z, d_m, h_m \right)$ and $\underline{\underline{S}}_{2} \left( y, z, d_m, h_m \right)$ are Sommerfeld integrals $\underline{S}_{1} \left( y, z, d_m, h_m, \nu \right)$ and $\underline{S}_{2} \left( y, z, d_m, h_m, \nu \right)$ based on the quasi-TEM assumption and are given by:

<p style="margin-bottom: 30px;"></p>

$$
\underline{\underline{S}}_{1} \left( y, z, d_m, h_m \right) = \int_{0}^{+\infty} \frac{e^{-(z+h_m){\lambda}}} {\lambda + \sqrt{\lambda^2 + k_{a}^2 - k_{e}^2}} \cos[(\mathbf{y}-d_{m})\lambda] d\lambda
$$

<p style="margin-bottom: 30px;"></p>

$$
\underline{\underline{S}}_{2} \left( y, z, d_m, h_m \right) = \int_{0}^{+\infty} \frac{e^{-(z+h_m){\lambda}}} {k_{e}^2 k_{a}^{-2} \lambda + \sqrt{\lambda^2 + k_{a}^2 - k_{e}^2}} \cos[(\mathbf{y}-d_{m})\lambda] d\lambda
$$

<p style="margin-bottom: 30px;"></p>


$$
d_{nm} \left( y, z, d_m, h_m \right) = \sqrt{\left( y - d_m \right)^2 + \left( z - h_m \right)^2}
$$

<p style="margin-bottom: 30px;"></p>

$$
D_{nm} \left( y, z, d_m, h_m \right) = \sqrt{\left( y - d_m \right)^2 + \left( z + h_m \right)^2}
$$

<p style="margin-bottom: 30px;"></p>

$$
k_{a}^2 = -j \omega \mu _0 j \omega \varepsilon _0, \hspace{0.3cm} k_{e}^2 = -j \omega \mu _1 \left( \sigma _1 + j \omega \varepsilon _1 \right)
$$

## Python Class PerUnitParameters

In [28]:
import numpy as np
from scipy.special import iv
from scipy.constants import mu_0, epsilon_0
from scipy.integrate import quad

class PerUnitParameters(Multiconductor):
    """ This class contains the basic geometry of the system. """

    def __init__(self, mtl, f, sigma_1, er_1=1, mur_1=1, ge=0):
        """ This method initializes the class. """
        super().__init__(mtl)

        # Soil Permittivity [np.array]
        self.er_1 = er_1

        # Soil conductivity [np.array]
        self.sigma_1 = sigma_1

        # Soil Permeability [np.array]
        self.mur_1 = mur_1

        # External Conductance [np.array]
        self.ge = ge

        # Angular frequency (rad/s)
        self.jw = 1j * 2 * np.pi * f

        # Constant vacuum terms
        self.jw_mu0__2pi = self.jw * mu_0 / 2 / np.pi
        self.jw_2pi_e0 = self.jw * 2 * np.pi * epsilon_0

        # Air wave number (rad/m)
        self.ka2 = - self.jw * mu_0 * self.jw * epsilon_0

        # Earth wave number (rad/m)
        self.ke2 = - self.jw * self.mur_1 * mu_0 * \
            (self.sigma_1 + self.jw * self.er_1 * epsilon_0)
    

### Series Impedance Matrix, $\mathbf{Z_s}$

The series or longitudinal impedance matrix,  $\mathbf{Z_s}$, is computed from the geometric and electric characteristics of the transmission line. In general, it can be decomposed into two terms: 

$$
\mathbf{Z_s}(\omega) = \mathbf{Z_i}(\omega) + \mathbf{Z_e}(\omega) = \mathbf{R_s}(\omega) + j\omega \mathbf{L}(\omega)
$$

where,  
$\mathbf{Z_e}$ and $\mathbf{Z_i}$ are respectively the external and the internal series impedance matrix. 

$\mathbf{Z_s}$ is a complex and symmetric matrix, whose elements are frequency dependent. The external impedance accounts for the magnetic field exterior to the conductor and comprises the contributions of the magnetic field in the air and the field penetrating the earth.

#### Internal Series Impedance  

The internal impedance of a round wire is found from the total current in the wire and the electric field intensity at the surface (surface impedance):

$$
Z_i = \frac{E \big|_{\text{surface}}}{I_{\text{total}}} = - \frac{Z_{\text{cw}}}{2 \pi r_c} \frac{I_0(\gamma_c r_c)}{I_1(\gamma_c r_c)} 
$$

where  
$I_0(.)$ and $I_1(.)$ are the modified Bessel functions,  
$Z_{\text{cw}}$ is the wave impedance in the conductor given by

$$
Z_{\text{cw}} = \sqrt{\frac{j \omega \mu_c}{\sigma_c + j \omega \varepsilon_c}}
$$

$\gamma_c$ is the propagation constant in the conducting material

$$
\gamma_c = \sqrt{j \omega \mu_c (\sigma_c + j \omega \varepsilon_c)}
$$

The conductivity, permittivity, permeability, and the radius of the conductor are denoted by $\sigma_c$, $\varepsilon_c$, $\mu_c$, and $r_c$, respectively.

* Low frequencies analysis $\left| \gamma_c rc \right| << 1$  

    Using series expansions of the Bessel functions the internal impedance is

    \begin{equation}
    Z_{LF} \approx R_{dc} \left[ {1 + \frac{1}{48} \left( \frac{r_c}{\delta} \right)^2} \right] + j \frac{\omega \mu}{8 \pi}
    \end{equation}

    where  

    \begin{equation}
    \delta = \sqrt {\frac {2}{\omega \mu_c \sigma_c}}
    \end{equation}

    is the skin depth in the conductor. The second term in the brackets is a correction useful for $r_c/\delta$ as large as unity, while the imaginary term corresponds to a low-frequency internal inductance and $R_{dc}$ is the direct current resistance given by

    \begin{equation}
    R_{dc} = \frac {1}{\pi r_c^2 \sigma_c}
    \end{equation}

* High frequencies analysis $\left| \gamma_c rc \right| >> 1$  

    Using asymptotic expressions of the Bessel functions, the internal impedance becomes

    \begin{equation}
    Z_{HF} \approx \frac{1}{2 \pi r_c \sigma_c} \frac{1}{\dot{p}_c}
    \end{equation}

    where $\dot{p}_c$ is the complex penetration depth for the conductor and is given by

    \begin{equation}
    \dot{p}_c = \sqrt {\frac{1}{j \omega \mu_c \sigma_c}}
    \end{equation}

Using different approximations for different frequency ranges produces discontinuities in the calculated impedance. To avoid such discontinuities the following expression can be used for the whole frequency range:

\begin{equation}
Z_i = \sqrt{R_{dc}^2 + Z_{HF}^2}
\end{equation}

Finally, the internal impedance matrix for a multiconductor line with $n$ phases is defined as follows:

$$

\begin{equation}

Z_{i} = 
\begin{bmatrix}
Z_{i,1} & 0       & \cdots & 0 \\
0       & Z_{i,2} & \cdots & 0 \\
\vdots  & \vdots  & \ddots & \vdots \\
0       & 0       & \cdots & Z_{i,n}
\end{bmatrix}

\end{equation}


In [24]:
def internal_impedance(self, type_form='approx'):
        """ This method calculates the internal impedance of solid wires. """

        # Impedance matrix
        N = len(self.surfaces)  # pylint: disable=invalid-name
        Zi = np.zeros((N, N), dtype=complex)  # pylint: disable=invalid-name
        Ri_cc = np.zeros_like(Zi)  # pylint: disable=invalid-name
        Zi_hf = np.zeros_like(Zi)  # pylint: disable=invalid-name

        # Loop over the conductors
        for p, conductor in enumerate(self.surfaces):

            # Angular frequency (rad/s)
            jw_mu = self.jw * self.mu[p]

            # Outer wire radius (m)
            ro = conductor['radius']

            # Wire conductivity (S/m)
            sigma = self.sigma[p]

            # High frequency impedance (ohm/m)
            # Derived assuming current conduction in a ring with
            # thickness equal to the penetration depth (ohm/m).
            zi_hf = 1.0/(2 * np.pi * ro) * np.sqrt(jw_mu/sigma)

            # Approximation Closed-Form Expression
            if type_form == 'approx':

                # Continuous current resistance (ohm/m)
                ri_cc = 1.0/(sigma * np.pi * ro**2)

                Ri_cc[p, p] = ri_cc
                Zi_hf[p, p] = zi_hf
                Zi[p, p] = np.sqrt(ri_cc ** 2 + zi_hf ** 2)

            else:
                arg = np.sqrt(jw_mu * sigma) * ro
                io = iv(0, arg)
                i1 = iv(1, arg)
                Zi[p, p] = zi_hf * (io/i1)

        return Zi, Ri_cc, Zi_hf

# Add the method to the PerUnitParameters class
PerUnitParameters.internal_impedance = internal_impedance

### External Impedance, $Z_e$

In [25]:
def external_impedance(self):
        """ This method calculates the impedance matrix of the earth return path. """

        # Impedance matrix
        N = len(self.surfaces)  # pylint: disable=invalid-name
        Ze = np.zeros((N, N), dtype=complex)  # pylint: disable=invalid-name

        # Loop over the conductors
        for n, _ in enumerate(self.surfaces):
            for m, _ in enumerate(self.surfaces):

                # Distance between the conductors (m)
                dnm, Dnm, _, _ = self.conductor_distances(n, m)  # pylint: disable=invalid-name

                # External impedance (ohm/m)
                Ze[n, m] = self.jw_mu0__2pi * np.log(Dnm/dnm)

        return Ze

# Add the method to the PerUnitParameters class
PerUnitParameters.external_impedance = external_impedance